# OpenAI API Examples

Companion notebook for [`README.md`](./README.md).  
All examples use the `openai` library and load credentials from a `.env` file.

**Requirements:** `conda activate agents` and `pip install openai python-dotenv numpy`

In [6]:
# Shared setup — run this cell first
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()          # reads OPENAI_API_KEY from .env
client = OpenAI()      # picks up the key automatically
print("Client ready.")

Client ready.


---
## 1. Basic Chat Completion

Every request is a list of `messages` with roles `system`, `user`, or `assistant`.

In [7]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user",   "content": "What is the capital of Japan?"},
    ],
)

print(response.choices[0].message.content)
print(f"\nTokens used: {response.usage.total_tokens}")

The capital of Japan is Tokyo.

Tokens used: 31


---
## 2. Multi-Turn Conversation

Maintain context by appending each assistant reply to the `messages` list.

In [8]:
messages = [{"role": "system", "content": "You are a helpful assistant."}]

def chat(user_input: str) -> str:
    messages.append({"role": "user", "content": user_input})
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    return reply

print(chat("My name is Alice."))
print()
print(chat("What is my name?"))   # model should remember

Hello, Alice! How can I assist you today?

Your name is Alice.


---
## 3. Streaming

### 3a. Simple `stream=True`

Receive tokens incrementally as they are generated.

In [9]:
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Write a haiku about Python."}],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

Lines of logic flow,  
Indentations guide the way,  
Code like poetry.


### 3b. `.stream()` context manager

Richer event types; access to the final completion object with token usage.

In [10]:
with client.chat.completions.stream(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "List three benefits of Python."}],
    stream_options={"include_usage": True},
) as stream:
    for event in stream:
        if event.type == "content.delta":
            print(event.delta, end="", flush=True)

completion = stream.get_final_completion()
print(f"\n\nTotal tokens: {completion.usage.total_tokens}")

Python offers numerous benefits, but here are three key advantages:

1. **Ease of Learning and Readability**: Python has a simple and clean syntax that resembles natural language, making it accessible for beginners. This readability allows developers to write code more quickly and maintain it more efficiently, facilitating collaboration among teams.

2. **Wide Range of Libraries and Frameworks**: Python boasts a rich ecosystem of libraries and frameworks, such as NumPy, Pandas, Django, and TensorFlow. These tools enable developers to work on various applications, from data analysis and web development to machine learning and artificial intelligence, without needing to build everything from scratch.

3. **Cross-Platform Compatibility**: Python is a cross-platform language, which means that code written in Python can run on various operating systems, including Windows, macOS, and Linux, without requiring significant modifications. This versatility makes it a popular choice for developers

---
## 4. Structured Output with Pydantic

Use `.parse()` with a Pydantic model as `response_format`.  
The SDK converts it to JSON schema, sends it to the API, and parses the reply back into typed Python objects.

> Requires `gpt-4o-2024-08-06` or later.

In [11]:
from typing import List
from pydantic import BaseModel

class Step(BaseModel):
    explanation: str
    output: str

class MathResponse(BaseModel):
    steps: List[Step]
    final_answer: str

completion = client.chat.completions.parse(
    model="gpt-4o-2024-08-06",
    messages=[
        {"role": "system", "content": "You are a math tutor."},
        {"role": "user",   "content": "Solve: 8x + 31 = 2"},
    ],
    response_format=MathResponse,
)

message = completion.choices[0].message
if message.parsed:
    for step in message.parsed.steps:
        print(f"{step.explanation}  →  {step.output}")
    print("\nAnswer:", message.parsed.final_answer)
else:
    print("Refusal:", message.refusal)

We start with the given equation: \(8x + 31 = 2\). Our goal is to solve for \(x\). The first step is to isolate the term with \(x\) by subtracting 31 from both sides of the equation.  →  8x + 31 - 31 = 2 - 31
Simplify both sides: On the left side, the \(+31\) and \(-31\) cancel out, and on the right side, we perform the subtraction.  →  8x = -29
Next, we isolate \(x\) by dividing both sides of the equation by 8.  →  x = \frac{-29}{8}

Answer: x = \frac{-29}{8}


---
## 5. Tool Use / Function Calling

1. Define tool schemas.  
2. First call: model decides whether to call a tool.  
3. Execute the function locally.  
4. Second call: model produces the final answer using the tool result.

In [12]:
import json

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Return current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["city"],
            },
        },
    }
]

messages = [{"role": "user", "content": "What's the weather in Tokyo?"}]

# First call
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    tool_choice="auto",
)

message = response.choices[0].message
print("Finish reason:", response.choices[0].finish_reason)

if message.tool_calls:
    tool_call = message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    print("Tool called:", tool_call.function.name, "|", args)

    # Simulate the actual function
    tool_result = {"temperature": 18, "condition": "Cloudy", "city": args["city"]}

    messages.append(message)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(tool_result),
    })

    # Second call — final answer
    final = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
    )
    print("\nFinal answer:", final.choices[0].message.content)

Finish reason: tool_calls
Tool called: get_weather | {'city': 'Tokyo'}

Final answer: The current weather in Tokyo is 18°C and cloudy.


---
## 6. Vision (Image Input)

Pass images alongside text in `content`. Supports public URLs and base64 local files.

### 6a. Image from URL

In [13]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text",      "text": "What animal is in this image?"},
                {"type": "image_url", "image_url": {
                    "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/2023_06_08_Raccoon1.jpg/400px-2023_06_08_Raccoon1.jpg"
                }},
            ],
        }
    ],
)
print(response.choices[0].message.content)

The animal in the image is a raccoon.


### 6b. Image from local file (base64)

In [14]:
import base64

# Replace with a real local image path to test
image_path = "./assets/2023_06_08_Raccoon1.jpg"
try:
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text",      "text": "Describe this image."},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                ],
            }
        ],
    )
    print(response.choices[0].message.content)
except FileNotFoundError:
    print(f"File not found: {image_path} — replace with a real image path to test.")

The image features a raccoon peeking out from behind a tree. The tree's bark is prominently displayed, showcasing its texture and patterns. The raccoon has distinct facial markings, including a black mask around its eyes and a bushy tail. The background is dark, which helps highlight the raccoon's features and creates a sense of intrigue as it gazes out from its hiding spot.


---
## 7. Embeddings

Convert text to a numerical vector for semantic search, clustering, or RAG.

In [15]:
import numpy as np

# Single embedding
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="The quick brown fox jumps over the lazy dog.",
)
vector = response.data[0].embedding
print(f"Dimensions: {len(vector)}")

Dimensions: 1536


In [16]:
# Batch embeddings + cosine similarity
texts = [
    "Machine learning is a branch of AI.",
    "Deep learning uses neural networks.",
    "I love pizza and pasta.",
]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts,
)
vectors = [np.array(item.embedding) for item in response.data]

def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"ML vs DL:    {cosine_similarity(vectors[0], vectors[1]):.3f}")
print(f"ML vs Food:  {cosine_similarity(vectors[0], vectors[2]):.3f}")

ML vs DL:    0.497
ML vs Food:  0.081


---
## 8. Async Client

Use `AsyncOpenAI` for non-blocking calls in async frameworks (FastAPI, asyncio).

> In a Jupyter notebook `asyncio.run()` is not needed — use `await` directly.

In [17]:
from openai import AsyncOpenAI

async_client = AsyncOpenAI()   # also reads OPENAI_API_KEY from env

# Basic async completion (await works directly in Jupyter)
response = await async_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Name three planets."}],
)
print(response.choices[0].message.content)

Three planets in our solar system are Earth, Mars, and Jupiter.


In [18]:
# Async streaming
stream = await async_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Count to five, one word per line."}],
    stream=True,
)

async for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

One  
Two  
Three  
Four  
Five  


---
## Model Reference

| Model | Context | Best for |
|---|---|---|
| `gpt-4o` | 128k | Multimodal, high accuracy |
| `gpt-4o-mini` | 128k | Fast, cost-efficient |
| `gpt-4o-2024-08-06` | 128k | Structured output, tool use |
| `o3` | 200k | Complex reasoning |
| `o4-mini` | 200k | Fast reasoning |
| `text-embedding-3-small` | — | Embeddings (cheap) |
| `text-embedding-3-large` | — | Embeddings (accurate) |

> Check [platform.openai.com/docs/models](https://platform.openai.com/docs/models) for the latest list.